### 대화 엔티티 메모리

- 절대적인 양 기준이 아닌 대화에서 특정 엔티티에 대한 주어진 사실을 기억하는 메모리
- **대화 엔티티 메모리**는 대화에서 특정 엔티티에 대한 주어진 사실을 기억
- **엔티티**란 대화나 데이터에서 특정한 의미를 가지는 핵심 정보를 의미
- 엔티티 메모리는 엔티티에 대한 정보를 추출하고(LLM 사용) 시간이 지남에 따라 해당 엔티티에 대한 지식을 축적(역시 LLM 사용).

In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationEntityMemory
from langchain.memory.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE

- Entity 메모리를 효과적으로 사용하기 위하여, 제공되는 프롬프트를 사용
- 과거에 했던 대화 내용 중에서 중요한 정보들을 추출해 내는 프롬프트 템플릿

In [4]:
print(ENTITY_MEMORY_CONVERSATION_TEMPLATE.template)

You are an assistant to a human, powered by a large language model trained by OpenAI.

You are designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, you are able to generate human-like text based on the input you receive, allowing you to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

You are constantly learning and improving, and your capabilities are constantly evolving. You are able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. You have access to some personalized information provided by the human in the Context section below. Additionally, you are able to generate your own text based on the input you receive, allowing you to engage in discussions and provide explanations and de

- **ConversationChain**은 LangChain에서 대화 흐름을 관리하는 객체
- LLM과 대화 메모리를 결합하여 대화의 컨텍스트를 유지

In [6]:
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

conversation = ConversationChain(
    llm=llm,
    prompt=ENTITY_MEMORY_CONVERSATION_TEMPLATE,
    memory=ConversationEntityMemory(llm=llm),
)

- 입력한 대화를 바탕으로 `ConversationEntityMemory` 는 주요 Entity 정보를 별도로 저장

In [7]:
conversation.predict(
    input="테디와 셜리는 한 회사에서 일하는 동료입니다."
    "테디는 개발자이고 셜리는 디자이너입니다. "
    "그들은 최근 회사에서 일하는 것을 그만두고 자신들의 회사를 차릴 계획을 세우고 있습니다."
)

'그렇군요! 테디와 셜리가 함께 회사를 차리기로 한 결정은 정말 흥미로운데요. 그들이 어떤 분야에서 사업을 시작하려고 하는지, 그리고 그들의 비전이나 목표가 무엇인지 궁금합니다. 혹시 그들이 어떤 계획을 가지고 있는지 더 자세히 알고 계신가요?'

- `entity_store.store` 에서 Entity 정보를 확인 가능
- 대화 내용을 그대로 저장하지 않고, 중요한 정보를 추출해서 저장

In [8]:
conversation.memory.entity_store.store

{'테디': '테디는 개발자이며, 셜리와 함께 자신들의 회사를 차릴 계획을 세우고 있습니다.',
 '셜리': '셜리는 디자이너로, 테디와 함께 자신들의 회사를 차릴 계획을 세우고 있습니다.'}